# Interconnectedness Analysis

How different mapping types contribute to connectivity in the LOD cloud.

In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

OUTPUT_DIR = Path("/home/javier.millanacosta/rdfsolve/output")
MAPPINGS_DIR = OUTPUT_DIR / "mappings"
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
def parse_sssom_tsv(filepath: Path) -> tuple[dict, list[dict]]:
    """Parse SSSOM TSV file."""
    metadata = {}
    mappings = []
    headers = []
    
    with filepath.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("#"):
                content = line[1:].strip()
                if ":" in content:
                    key, _, value = content.partition(":")
                    metadata[key.strip()] = value.strip()
            elif not headers:
                headers = line.split("\t")
            else:
                values = line.split("\t")
                mappings.append(dict(zip(headers, values)))
    
    return metadata, mappings

In [ ]:
# Load schemas to get dataset list
datasets = set()
for f in OUTPUT_DIR.glob("**/*_schema.jsonld"):
    datasets.add(f.parent.name)

print(f"Found {len(datasets)} datasets with schemas")

In [ ]:
# Build connectivity graph from SSSOM mappings
# Edges: dataset pairs connected by mappings

def extract_dataset_from_source(source_uri: str) -> str:
    """Extract dataset name from VoID URI."""
    if "/dataset/" in source_uri:
        return source_uri.split("/dataset/")[-1].strip()
    return source_uri

# Track edges by mapping type
edges_by_type = defaultdict(lambda: defaultdict(int))

for sssom_file in MAPPINGS_DIR.glob("**/*.sssom.tsv"):
    metadata, mappings = parse_sssom_tsv(sssom_file)
    
    subj_source = metadata.get("subject_source", "")
    obj_source = metadata.get("object_source", "")
    
    if subj_source and obj_source:
        ds1 = extract_dataset_from_source(subj_source)
        ds2 = extract_dataset_from_source(obj_source)
        
        if ds1 and ds2 and ds1 != ds2:
            pair = tuple(sorted([ds1, ds2]))
            
            # Categorize by file location
            if "enriched" in str(sssom_file):
                mapping_type = "enriched_external"
            elif "schema-patterns" in sssom_file.name:
                mapping_type = "schema_patterns"
            else:
                mapping_type = "other"
            
            edges_by_type[mapping_type][pair] += len(mappings)

print("Connectivity by mapping type:")
for mtype, edges in edges_by_type.items():
    total_mappings = sum(edges.values())
    print(f"  {mtype}: {len(edges)} dataset pairs, {total_mappings:,} mappings")

In [ ]:
# Find all connected datasets
all_edges = defaultdict(int)
for mtype, edges in edges_by_type.items():
    for pair, count in edges.items():
        all_edges[pair] += count

connected_datasets = set()
for ds1, ds2 in all_edges.keys():
    connected_datasets.add(ds1)
    connected_datasets.add(ds2)

print(f"Connected datasets: {len(connected_datasets)}")
print(f"Total unique edges: {len(all_edges)}")
print(f"Total mappings: {sum(all_edges.values()):,}")

In [ ]:
# Top dataset pairs by connectivity
print("Top 20 most connected dataset pairs:")
sorted_edges = sorted(all_edges.items(), key=lambda x: x[1], reverse=True)
for (ds1, ds2), count in sorted_edges[:20]:
    print(f"  {count:6d}  {ds1} <-> {ds2}")

In [ ]:
# Dataset degree (number of connections)
degree = Counter()
for ds1, ds2 in all_edges.keys():
    degree[ds1] += 1
    degree[ds2] += 1

print("Top 20 most connected datasets:")
for ds, deg in degree.most_common(20):
    print(f"  {deg:3d} connections  {ds}")

In [ ]:
# Contribution of each mapping type to overall connectivity
print("\nContribution by mapping type:")
print("=" * 60)

total_pairs = len(all_edges)
total_mappings = sum(all_edges.values())

for mtype, edges in sorted(edges_by_type.items()):
    pairs = len(edges)
    mappings = sum(edges.values())
    
    # Find unique pairs contributed by this type
    unique_pairs = set(edges.keys())
    other_pairs = set()
    for other_type, other_edges in edges_by_type.items():
        if other_type != mtype:
            other_pairs.update(other_edges.keys())
    
    exclusive_pairs = unique_pairs - other_pairs
    
    print(f"\n{mtype}:")
    print(f"  Dataset pairs: {pairs} ({pairs/total_pairs*100:.1f}% of total)")
    print(f"  Mappings: {mappings:,} ({mappings/total_mappings*100:.1f}% of total)")
    print(f"  Exclusive pairs: {len(exclusive_pairs)} (only connected via this type)")

In [ ]:
# Summary statistics
print("\nSummary:")
print("=" * 60)
print(f"Total datasets in LOD cloud: {len(datasets)}")
print(f"Datasets with SSSOM connections: {len(connected_datasets)}")
print(f"Coverage: {len(connected_datasets)/len(datasets)*100:.1f}%")
print(f"Total dataset pairs connected: {len(all_edges)}")
print(f"Total class mappings: {sum(all_edges.values()):,}")